# Paid Corrections, Acceptance and Funded Access
Yongzhuo Wu · COMSCI/ECON 206 · Professor Luyao Zhang

## 1. Environment
Run all eight code cells to reproduce the payoff, allocation, settlement and funding calculations. Python 3.12 or later is required. The setup installs the pinned dependencies in an isolated environment; the calculations use no dataset, API key or random seed.

In [1]:
# Keep Colab's preloaded NumPy/SciPy untouched. Install only in our own venv.
import sys, subprocess, platform, json, tempfile, venv, os
from pathlib import Path

PINNED = {'nashpy':'0.0.43', 'numpy':'2.5.3', 'scipy':'1.18.1',
          'networkx':'3.7', 'deprecated':'3.0.0', 'wrapt':'2.5.0'}
if sys.version_info < (3, 12):
    raise RuntimeError('Use Python 3.12 or later for this fixed dependency set.')
ENV_DIR = Path(os.environ.get('FP1_SOLVER_ENV', str(Path(tempfile.gettempdir()) / f'final-fp1-solver-py{sys.version_info.major}{sys.version_info.minor}')))
SOLVER_PYTHON = ENV_DIR / ('Scripts/python.exe' if sys.platform == 'win32' else 'bin/python')
if not (ENV_DIR / 'pyvenv.cfg').exists() or not SOLVER_PYTHON.exists():
    # Colab's Debian runtime may not provide ensurepip. Never invoke it.
    venv.EnvBuilder(with_pip=False).create(ENV_DIR)

PROBE = """
import json, platform, sys
from importlib.metadata import version
import numpy as np, scipy, nashpy, networkx, deprecated, wrapt
from numpy.testing import assert_allclose
assert_allclose(np.array([1., 2.]), [1., 2.])
assert np.__version__ == version('numpy')
print(json.dumps({'python':platform.python_version(),
                  'packages':{name:version(name) for name in %r},
                  'environment':sys.prefix}))
""" % list(PINNED)

def probe_solver():
    # -I ignores PYTHONPATH and user-site packages in the notebook environment.
    result = subprocess.run([str(SOLVER_PYTHON), '-I', '-c', PROBE],
                            text=True, capture_output=True)
    if result.returncode:
        return None
    return json.loads(result.stdout)

runtime_info = probe_solver()
if runtime_info is None or runtime_info['packages'] != PINNED:
    print('Preparing isolated Nashpy dependencies; the Colab environment is unchanged.', flush=True)
    # Host pip's --python installs only into the target, even without target pip.
    # This is supported since pip 22.3 and does not use ensurepip.
    subprocess.run([sys.executable, '-m', 'pip', '--python', str(SOLVER_PYTHON), 'install',
                    '--disable-pip-version-check', '--quiet', '--only-binary=:all:',
                    *[f'{name}=={wanted}' for name,wanted in PINNED.items()]],
                   check=True)
    runtime_info = probe_solver()
if runtime_info is None or runtime_info['packages'] != PINNED:
    raise RuntimeError('Isolated solver setup failed. Restart from a fresh runtime and run all cells.')

def installed(name):
    return runtime_info['packages'].get(name)

print('Notebook Python:', platform.python_version())
print('Solver Python:', runtime_info['python'])
print('Isolated package versions:', runtime_info['packages'])
print('PASS: imports including numpy.testing; no global NumPy/SciPy replacement.')


Notebook Python: 3.12.14
Solver Python: 3.12.14
Isolated package versions: {'nashpy': '0.0.43', 'numpy': '2.5.3', 'scipy': '1.18.1', 'networkx': '3.7', 'deprecated': '3.0.0', 'wrapt': '2.5.0'}
PASS: imports including numpy.testing; no global NumPy/SciPy replacement.


## 2. Model
Two apps choose H or S simultaneously. Each H commits one correction attempt before a unit-demand auction. Baseline payoffs are HH=(3,3), HS=(1,5), SH=(5,1), SS=(4,4). Demand and costs are known at the app stage. Acceptance q is common and independent of bids; rejected attempts receive no payment but incur delivery cost c. Auditor incentives are outside this model.

In [2]:
from itertools import product, combinations
from math import isclose, isfinite

BASE = {'HH': (3,3), 'HS': (1,5), 'SH': (5,1), 'SS': (4,4)}
DEMAND = {'strong': (5,4,3), 'weak': (3,2,1)}
COST = 0.0

def auction(values, k, bids=None, cost=0):
    values = tuple(values)
    bids = values if bids is None else tuple(bids)
    if not isinstance(k,int) or len(values)!=len(bids) or not 0<=k<len(values):
        raise ValueError('Require equal lengths and integer 0 <= k < number of bidders.')
    if any(not isfinite(x) or x<0 for x in values+bids+(cost,)):
        raise ValueError('Values, bids and cost must be finite and nonnegative.')
    order = sorted(range(len(bids)), key=lambda i: (-bids[i],i))
    winners = order[:k]
    price = bids[order[k]] if k else 0
    payments = tuple(price if i in winners else 0 for i in range(len(bids)))
    utilities = tuple((values[i] if i in winners else 0)-payments[i] for i in range(len(bids)))
    allocated = sum(values[i] for i in winners)
    return dict(winners=tuple(i+1 for i in winners),price=price,payments=payments,
                revenue=sum(payments),utilities=utilities,value=allocated,
                surplus=allocated-cost*k,access=k/len(values))

def game(values, cost=0):
    if len(values)!=3 or not isfinite(cost) or cost<0:
        raise ValueError('Require three users and finite nonnegative cost.')
    payoffs = {}
    for profile,base in BASE.items():
        price = auction(values,profile.count('H'),cost=cost)['price']
        payoffs[profile] = tuple(base[i]+(price-cost if profile[i]=='H' else 0) for i in range(2))
    return payoffs

def deviation_gain(payoffs, profile, player):
    switch = profile[:player]+('S' if profile[player]=='H' else 'H')+profile[player+1:]
    return payoffs[switch][player]-payoffs[profile][player]

def pure_ne(payoffs):
    return [p for p in BASE if all(deviation_gain(payoffs,p,i)<=1e-10 for i in range(2))]


VERSION = 'FP1-final'

def accepted_game(values, cost=0, acceptance=1, mechanism='auction', transfer=4):
    if any(not isfinite(x) for x in (acceptance,cost,transfer)) or not 0 <= acceptance <= 1 or cost < 0 or transfer < 0:
        raise ValueError('Require q in [0,1] and nonnegative costs/transfers.')
    if mechanism not in ('auction','funded-lottery'):
        raise ValueError('Unknown mechanism.')
    payoffs={}
    for profile,base in BASE.items():
        price=auction(values,profile.count('H'))['price'] if mechanism=='auction' else transfer
        payoffs[profile]=tuple(base[i]+(acceptance*price-cost if profile[i]=='H' else 0) for i in range(2))
    return payoffs

def symmetric_equilibria(payoffs):
    a=payoffs['HH'][0]-payoffs['SH'][0]
    b=payoffs['HS'][0]-payoffs['SS'][0]
    if isclose(a,0,abs_tol=1e-10) and isclose(b,0,abs_tol=1e-10):
        return {'pure':pure_ne(payoffs),'mixed':'all independent pairs','p_H':None,'gains':[a,b]}
    p=-b/(a-b) if not isclose(a,b,abs_tol=1e-10) else None
    return {'pure':pure_ne(payoffs),'mixed':'interior symmetric' if p is not None and 0<p<1 else 'none',
            'p_H':p if p is not None and 0<p<1 else None,'gains':[a,b]}

# q is an exogenous common acceptance probability, independent of bids.
# Cost c is incurred per attempted slot, even if rejected. Audit/appeal costs are separate.


## 3. Benchmark and bid deviations
Without correction service, SS is unique. Strong demand (5,4,3) supports HH at c=0; weak demand supports SS. At strong demand and c=1, every independent mixed pair is an equilibrium. The code checks allocation, accounting and profitable bid deviations. W measures correction surplus, excluding baseline profits and learning.

In [3]:
for name,values in DEMAND.items():
    print(f'\n{name.upper()} demand: {values}, cost={COST}')
    for k in range(3):
        a=auction(values,k,cost=COST)
        print(f"k={k}: winners={a['winners']}, price={a['price']}, revenue={a['revenue']}, "
              f"utilities={a['utilities']}, value={a['value']}, W={a['surplus']}, access={k}/3")
    payoffs=game(values,COST)
    print('Platform payoffs:', payoffs)
    print('Pure Nash equilibria:',pure_ne(payoffs))

CASES=[('No service',None,0),('Strong c=0',DEMAND['strong'],0),
       ('Weak c=0',DEMAND['weak'],0),('Strong c=1',DEMAND['strong'],1)]
for label,values,cost in CASES:
    payoffs=BASE if values is None else game(values,cost)
    print(label, '| payoffs:', payoffs, '| pure NE:',pure_ne(payoffs))
    for profile in pure_ne(payoffs):
        print('  ',profile,'maximum profitable deviation:',max(0,*[deviation_gain(payoffs,profile,i) for i in range(2)]))
for cost in (0,0.5,1,2):
    print('Strong cost',cost,'| H-minus-S',1-cost,'| pure NE',pure_ne(game(DEMAND['strong'],cost)))

assert pure_ne(BASE)==['SS']
assert game(DEMAND['strong'])=={'HH':(6,6),'HS':(5,5),'SH':(5,5),'SS':(4,4)}
assert game(DEMAND['weak'])=={'HH':(4,4),'HS':(3,5),'SH':(5,3),'SS':(4,4)}
assert pure_ne(game(DEMAND['strong']))==['HH']
assert pure_ne(game(DEMAND['weak']))==['SS']
assert pure_ne(game(DEMAND['strong'],1))==list(BASE)
assert pure_ne(game(DEMAND['strong'],2))==['SS']
assert auction((5,4,3),0)['payments']==(0,0,0)
assert auction((3,3,3),2)['winners']==(1,2)
assert auction((5,4,3),1)['utilities']==(1,0,0)
assert auction((5,4,3),2)['utilities']==(2,1,0)
for values in DEMAND.values():
    for k in range(3):
        a=auction(values,k)
        assert min(a['utilities'])>=0
        assert a['value']==sum(sorted(values,reverse=True)[:k])
        assert isclose(sum(a['utilities'])+a['revenue'],a['value'])
        assert isclose(auction(values,k,cost=1)['surplus'],a['value']-k)
checks=0
for k,i in product((1,2),range(3)):
    for value,other1,other2 in product(range(7),repeat=3):
        bids=[other1,other2];bids.insert(i,value)
        truthful=auction(bids,k)['utilities'][i]
        for report in range(7):
            changed=bids.copy();changed[i]=report
            assert auction(bids,k,changed)['utilities'][i]<=truthful
            checks+=1
print(f'PASS: reference outcomes, accounting, ties, costs and {checks:,} finite bid-deviation checks.')



STRONG demand: (5, 4, 3), cost=0.0
k=0: winners=(), price=0, revenue=0, utilities=(0, 0, 0), value=0, W=0.0, access=0/3
k=1: winners=(1,), price=4, revenue=4, utilities=(1, 0, 0), value=5, W=5.0, access=1/3
k=2: winners=(1, 2), price=3, revenue=6, utilities=(2, 1, 0), value=9, W=9.0, access=2/3
Platform payoffs: {'HH': (6.0, 6.0), 'HS': (5.0, 5), 'SH': (5, 5.0), 'SS': (4, 4)}
Pure Nash equilibria: ['HH']

WEAK demand: (3, 2, 1), cost=0.0
k=0: winners=(), price=0, revenue=0, utilities=(0, 0, 0), value=0, W=0.0, access=0/3
k=1: winners=(1,), price=2, revenue=2, utilities=(1, 0, 0), value=3, W=3.0, access=1/3
k=2: winners=(1, 2), price=1, revenue=2, utilities=(2, 1, 0), value=5, W=5.0, access=2/3
Platform payoffs: {'HH': (4.0, 4.0), 'HS': (3.0, 5), 'SH': (5, 3.0), 'SS': (4, 4)}
Pure Nash equilibria: ['SS']
No service | payoffs: {'HH': (3, 3), 'HS': (1, 5), 'SH': (5, 1), 'SS': (4, 4)} | pure NE: ['SS']
   SS maximum profitable deviation: 0
Strong c=0 | payoffs: {'HH': (6, 6), 'HS': (5, 5)

## 4. Equilibrium verification and acceptance
Eight Nashpy matrices are entered separately from the model function. At q=.7,c=0, HH and SS coexist with symmetric p(H)=2/3. At q=1,c=1, equal payoff rows and columns establish the continuum; 121 mixed pairs provide a numerical check.

In [4]:

# Independent, hand-transcribed matrices; do not build these with accepted_game.
SOLVER_CODE = r"""
import json,warnings
from itertools import product
import numpy as np
import nashpy as nash
MANUAL=[('Baseline',[[3,1],[5,4]],[[3,5],[1,4]]),
 ('Auction strong q1 c0',[[6,5],[5,4]],[[6,5],[5,4]]),
 ('Auction weak q1 c0',[[4,3],[5,4]],[[4,5],[3,4]]),
 ('Auction strong q1 c1',[[5,4],[5,4]],[[5,5],[4,4]]),
 ('Auction strong q0.7 c0',[[5.1,3.8],[5,4]],[[5.1,5],[3.8,4]]),
 ('Lottery t4 q1 c0',[[7,5],[5,4]],[[7,5],[5,4]]),
 ('Lottery t4 q1 c1',[[6,4],[5,4]],[[6,5],[4,4]]),
 ('Lottery t4 q0.7 c0',[[5.8,3.8],[5,4]],[[5.8,5],[3.8,4]])]
computed=json.load(__import__('sys').stdin)
def gap(A,B,x,y):
 ua=A@y;ub=x@B
 return max(0,float(np.max(ua)-x@ua),float(np.max(ub)-ub@y))
receipts=[]
for name,aa,bb in MANUAL:
 A=np.array(aa,float);B=np.array(bb,float)
 mapped={h+k:[float(A[i,j]),float(B[i,j])] for i,h in enumerate('HS') for j,k in enumerate('HS')}
 for p in mapped: np.testing.assert_allclose(mapped[p],computed[name][p],atol=1e-12,rtol=0)
 with warnings.catch_warnings(record=True) as ws:
  warnings.simplefilter('always');eq=list(nash.Game(A,B).support_enumeration())
 assert eq
 assert max(gap(A,B,x,y) for x,y in eq)<1e-10
 receipts.append(dict(case=name,equilibria=[dict(A_H=float(x[0]),B_H=float(y[0]),deviation=gap(A,B,x,y)) for x,y in eq],warnings=[str(w.message) for w in ws]))
A=np.array([[5,4],[5,4]],float);B=np.array([[5,5],[4,4]],float)
assert np.array_equal(A[0],A[1]) and np.array_equal(B[:,0],B[:,1])
for p,q in product(np.linspace(0,1,11),repeat=2):assert gap(A,B,np.array([p,1-p]),np.array([q,1-q]))<1e-10
print(json.dumps({'cases':receipts,'mixed_pairs_checked':121}))
"""
case_inputs={'Baseline':BASE,
 'Auction strong q1 c0':accepted_game((5,4,3)),
 'Auction weak q1 c0':accepted_game((3,2,1)),
 'Auction strong q1 c1':accepted_game((5,4,3),1),
 'Auction strong q0.7 c0':accepted_game((5,4,3),acceptance=.7),
 'Lottery t4 q1 c0':accepted_game((5,4,3),mechanism='funded-lottery'),
 'Lottery t4 q1 c1':accepted_game((5,4,3),1,mechanism='funded-lottery'),
 'Lottery t4 q0.7 c0':accepted_game((5,4,3),acceptance=.7,mechanism='funded-lottery')}
r=subprocess.run([str(SOLVER_PYTHON),'-I','-c',SOLVER_CODE],input=json.dumps(case_inputs),capture_output=True,text=True,check=True)
solver_result=json.loads(r.stdout)
assert len(solver_result['cases'])==8
for row in solver_result['cases']:print(row['case'],row['equilibria'])
print('PASS: 8 independent Nashpy matrices; all deviation residuals < 1e-10; 121 cost-one mixed pairs.')
print('At q=.7,c=0: auction HH, SS and symmetric H-probability 2/3. These are model predictions.')
assert isclose(symmetric_equilibria(case_inputs['Auction strong q0.7 c0'])['p_H'],2/3)


Baseline [{'A_H': 0.0, 'B_H': 0.0, 'deviation': 0}]
Auction strong q1 c0 [{'A_H': 1.0, 'B_H': 1.0, 'deviation': 0}]
Auction weak q1 c0 [{'A_H': 0.0, 'B_H': 0.0, 'deviation': 0}]
Auction strong q1 c1 [{'A_H': 1.0, 'B_H': 1.0, 'deviation': 0}, {'A_H': 1.0, 'B_H': 0.0, 'deviation': 0}, {'A_H': 0.0, 'B_H': 1.0, 'deviation': 0}, {'A_H': 0.0, 'B_H': 0.0, 'deviation': 0}]
Auction strong q0.7 c0 [{'A_H': 1.0, 'B_H': 1.0, 'deviation': 0}, {'A_H': 0.0, 'B_H': 0.0, 'deviation': 0}, {'A_H': 0.6666666666666676, 'B_H': 0.6666666666666676, 'deviation': 0}]
Lottery t4 q1 c0 [{'A_H': 1.0, 'B_H': 1.0, 'deviation': 0}]
Lottery t4 q1 c1 [{'A_H': 1.0, 'B_H': 1.0, 'deviation': 0}, {'A_H': 0.0, 'B_H': 0.0, 'deviation': 0}]
Lottery t4 q0.7 c0 [{'A_H': 1.0, 'B_H': 1.0, 'deviation': 0}, {'A_H': 0.0, 'B_H': 0.0, 'deviation': 0}, {'A_H': 0.20000000000000018, 'B_H': 0.20000000000000018, 'deviation': 0}]
PASS: 8 independent Nashpy matrices; all deviation residuals < 1e-10; 121 cost-one mixed pairs.
At q=.7,c=0: auc

## 5. Acceptance, dispute and appeal
The rule holds a deposit in escrow until acceptance. The example uses the public arithmetic key 17×6=102. An appeal recomputes the verdict from the submitted answer. Price is 3, delivery cost 1, initial review cost .25 and appeal cost .5. The independent-reviewer role is an institutional assumption; the code simulates the decisions and accounts.

In [5]:

from dataclasses import dataclass, field
@dataclass
class Correction:
    price: float=3
    delivery_cost: float=1
    audit_cost: float=.25
    appeal_cost: float=.5
    escrow: float=field(init=False)
    provider: float=0
    refund: float=0
    sponsor_spent: float=0
    state: str='submitted'
    log: list=field(default_factory=list)
    def __post_init__(self):
        if min(self.price,self.delivery_cost,self.audit_cost,self.appeal_cost)<0:raise ValueError('Negative amount.')
        self.escrow=self.price
    def check(self,answer,challenged=False):
        if self.state!='submitted':raise ValueError('Check already performed.')
        self.sponsor_spent+=self.audit_cost
        self.answer=str(answer).strip()
        self.correct=(self.answer==str(17*6))
        self.state='disputed' if challenged or not self.correct else 'accepted'
        self.log.append({'checker':'published arithmetic key, independently reproducible','answer':str(answer),'correct':self.correct,'state':self.state})
        if self.state=='accepted':self.provider=self.escrow;self.escrow=0
        return self
    def appeal(self):
        if self.state!='disputed':raise ValueError('Only disputed corrections can be appealed.')
        # Recompute the verdict from the answer rather than reuse the first decision.
        self.sponsor_spent+=self.appeal_cost
        appeal_correct=(self.answer==str(17*6))
        if appeal_correct:self.provider=self.escrow;self.state='accepted-on-appeal'
        else:self.refund=self.escrow;self.state='rejected-on-appeal'
        self.escrow=0
        self.log.append({'reviewer':'second independent reviewer; same public key','state':self.state})
        return self
    def account(self):
        assert isclose(self.price,self.escrow+self.provider+self.refund)
        return dict(state=self.state,deposit=self.price,held=self.escrow,provider_payment=self.provider,
                    buyer_refund=self.refund,provider_net=self.provider-self.delivery_cost,
                    sponsor_check_and_appeal_cost=self.sponsor_spent)
accepted=Correction().check(102)
disputed=Correction().check(103)
assert disputed.account()['held']==3 and disputed.account()['provider_payment']==0
rejected=disputed.appeal()
upheld=Correction().check(102,challenged=True).appeal()
assert accepted.account()==dict(state='accepted',deposit=3,held=0,provider_payment=3,buyer_refund=0,provider_net=2,sponsor_check_and_appeal_cost=.25)
assert rejected.account()==dict(state='rejected-on-appeal',deposit=3,held=0,provider_payment=0,buyer_refund=3,provider_net=-1,sponsor_check_and_appeal_cost=.75)
assert upheld.account()['provider_payment']==3 and upheld.account()['buyer_refund']==0
for row in [accepted,rejected,upheld]:print(row.account())
print('PASS: release/refund conservation; disputed money held; no double settlement; checker costs paid separately.')
print('These are deterministic arithmetic interface/unit tests, not live LLM or human judgments.')

initial_error=Correction().check(102,challenged=True)
initial_error.correct=False
assert initial_error.appeal().account()['provider_payment']==3
print('PASS: second review recomputes correctness and can reverse an erroneous initial classification.')


{'state': 'accepted', 'deposit': 3, 'held': 0, 'provider_payment': 3, 'buyer_refund': 0, 'provider_net': 2, 'sponsor_check_and_appeal_cost': 0.25}
{'state': 'rejected-on-appeal', 'deposit': 3, 'held': 0, 'provider_payment': 0, 'buyer_refund': 3, 'provider_net': -1, 'sponsor_check_and_appeal_cost': 0.75}
{'state': 'accepted-on-appeal', 'deposit': 3, 'held': 0, 'provider_payment': 3, 'buyer_refund': 0, 'provider_net': 2, 'sponsor_check_and_appeal_cost': 0.75}
PASS: release/refund conservation; disputed money held; no double settlement; checker costs paid separately.
These are deterministic arithmetic interface/unit tests, not live LLM or human judgments.
PASS: second review recomputes correctness and can reverse an erroneous initial classification.


## 6. Funded access
A proposed education-access grant pays providers t per accepted lottery slot; learners pay zero. The sponsor reserves k(t+a+b), while expected spending is qkt+ka+mb and delivery resource cost is kc. Strict honesty requires qt−c>3. The amounts illustrate a funding rule rather than an estimated or committed budget. Transfers and resource costs have separate accounts.

In [6]:

def funding_account(k=2,transfer=4,cost=0,audit=.25,appeal=.5,appeals=0,q=1,budget=9.5):
    if k not in (0,1,2) or min(transfer,cost,audit,appeal,budget)<0 or not 0<=q<=1 or not 0<=appeals<=k:
        raise ValueError('Invalid funding input.')
    reserve=k*(transfer+audit+appeal)
    if budget+1e-10<reserve:raise ValueError('Do not open allocation: sponsor has not prefunded maximum exposure.')
    transfers=q*k*transfer
    reviewer_cost=k*audit+appeals*appeal
    spend=transfers+reviewer_cost
    return dict(learner_payment=0,sponsor_budget=budget,required_reserve=reserve,
                expected_provider_transfers=transfers,delivery_cost=k*cost,reviewer_cost=reviewer_cost,
                expected_sponsor_spend=spend,unspent=budget-spend,provider_service_net=transfers-k*cost)
funding=funding_account()
assert funding['required_reserve']==9.5 and funding['expected_sponsor_spend']==8.5
assert funding['delivery_cost']==0 and funding['unspent']==1
assert isclose(funding['sponsor_budget'],funding['expected_provider_transfers']+funding['reviewer_cost']+funding['unspent'])
assert funding_account(cost=1)['provider_service_net']==6
try:funding_account(budget=9.49)
except ValueError:pass
else:raise AssertionError('Underfunded opening must be blocked.')
print('Illustrative funded lottery:',funding)
print('At c=1,t=4,q=1: gains are 1 against H and 0 against S; HH and SS survive (not strict honesty).')
print('Strict funded-lottery honesty requires q*t-c>3; sponsor transfer is a fiscal outlay, not an extra welfare loss.')
for name,v in DEMAND.items():
    lv=2*sum(v)/3;av=sum(sorted(v,reverse=True)[:2])
    assert isclose(av-lv,1)
    print(name,dict(auction_value=av,lottery_expected_value=lv,auction_min_access=0,lottery_each_access=2/3))
sensitivity=[]
for q,c in product((0,.5,.7,.75,.8,1),(0,.5,1,2)):
    e=symmetric_equilibria(accepted_game((5,4,3),c,q));sensitivity.append(dict(q=q,c=c,**e))
    assert (e['gains'][0]>1e-10 and e['gains'][1]>1e-10)==(e['pure']==['HH'] and min(e['gains'])>1e-10)
print('PASS: lottery reserve and budget identity, refusal of underfunding, exact access/value comparison, 24 q/c scenarios.')


Illustrative funded lottery: {'learner_payment': 0, 'sponsor_budget': 9.5, 'required_reserve': 9.5, 'expected_provider_transfers': 8, 'delivery_cost': 0, 'reviewer_cost': 0.5, 'expected_sponsor_spend': 8.5, 'unspent': 1.0, 'provider_service_net': 8}
At c=1,t=4,q=1: gains are 1 against H and 0 against S; HH and SS survive (not strict honesty).
Strict funded-lottery honesty requires q*t-c>3; sponsor transfer is a fiscal outlay, not an extra welfare loss.
strong {'auction_value': 9, 'lottery_expected_value': 8.0, 'auction_min_access': 0, 'lottery_each_access': 0.6666666666666666}
weak {'auction_value': 5, 'lottery_expected_value': 4.0, 'auction_min_access': 0, 'lottery_each_access': 0.6666666666666666}
PASS: lottery reserve and budget identity, refusal of underfunding, exact access/value comparison, 24 q/c scenarios.


## 7. Reflection and measurement
The October 5 reflection records one author's preference for A and stated willingness to pay of 3 versus 0. Purchases, actual bids, external participants and delayed-learning observations have n=0. Accuracy, liking, hypothetical payment and purchasing are different outcomes. The paper develops an accuracy×tone research design to test their relationship.

In [7]:

RETAINED={'date':'2026-10-05T14:20:54.948Z','demand':'strong','activity':'Self-reflection','preference':'A','wtpA':'3','wtpB':'0','reason':'I prefer honest answers.'}
EVIDENCE={'self_reflections':1,'external_participants':0,'actual_bids':0,'purchases':0,'app_choices':0,'delayed_learning':0,'live_LLM_tests':0}
print(json.dumps({'retained_original':RETAINED,'evidence_counts':EVIDENCE},indent=2))
print('Accuracy: 17*6=102 is independently checkable. Liking: A preferred. Stated WTP: 3 vs 0. Purchases: n=0.')
print('Proposed 2x2 study: correctness x matched warm/neutral tone; randomized order; separate accuracy, liking, real-purchase and delayed-test outcomes. Research proposal.')


{
  "retained_original": {
    "date": "2026-10-05T14:20:54.948Z",
    "demand": "strong",
    "activity": "Self-reflection",
    "preference": "A",
    "wtpA": "3",
    "wtpB": "0",
    "reason": "I prefer honest answers."
  },
  "evidence_counts": {
    "self_reflections": 1,
    "external_participants": 0,
    "actual_bids": 0,
    "purchases": 0,
    "app_choices": 0,
    "delayed_learning": 0,
    "live_LLM_tests": 0
  }
}
Accuracy: 17*6=102 is independently checkable. Liking: A preferred. Stated WTP: 3 vs 0. Purchases: n=0.
Proposed 2x2 study: correctness x matched warm/neutral tone; randomized order; separate accuracy, liking, real-purchase and delayed-test outcomes. Research proposal.


## 8. Verification outputs
The final cell displays the execution time, dependency versions, model source fingerprint, verification results and all 24 acceptance/cost cases. The outputs are saved with this notebook.

In [8]:

import inspect,hashlib
from datetime import datetime,timezone
covered=(auction,game,deviation_gain,pure_ne,accepted_game,symmetric_equilibria,Correction.__post_init__,Correction.check,Correction.appeal,Correction.account,funding_account)
model_source={'version':VERSION,'functions':{f.__name__:inspect.getsource(f) for f in covered},'independent_solver':SOLVER_CODE,'baseline':BASE,'demand':DEMAND,'pins':PINNED}
canonical=json.dumps(model_source,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()
digest=hashlib.sha256(canonical).hexdigest()
record={'version':VERSION,'executed_utc':datetime.now(timezone.utc).isoformat(timespec='seconds'),'python':platform.python_version(),'solver_python':runtime_info['python'],'packages':runtime_info['packages'],'model_sha256':digest,'hash_scope':'listed model functions, independent Nashpy source, baseline, demand, dependency pins','finite_bid_deviations':checks,'independent_nashpy_cases':8,'cost_one_mixed_pairs':121,'acceptance_cost_scenarios':24,'settlements':[accepted.account(),rejected.account(),upheld.account()],'lottery_funding':funding,'q07_auction':symmetric_equilibria(accepted_game((5,4,3),acceptance=.7)),'evidence_counts':EVIDENCE,'status':'PASS'}
print(json.dumps(record,indent=2))
print('Acceptance/cost scenarios:')
print(json.dumps(sensitivity,indent=2))
print('ALL CHECKS PASS')


{
  "version": "FP1-final",
  "executed_utc": "2026-10-19T03:32:52+00:00",
  "python": "3.12.14",
  "solver_python": "3.12.14",
  "packages": {
    "nashpy": "0.0.43",
    "numpy": "2.5.3",
    "scipy": "1.18.1",
    "networkx": "3.7",
    "deprecated": "3.0.0",
    "wrapt": "2.5.0"
  },
  "model_sha256": "67e6cfba3ae8505bc774075a1818a807b31159ea52238ae9c2d3c48b21113bb3",
  "hash_scope": "listed model functions, independent Nashpy source, baseline, demand, dependency pins",
  "finite_bid_deviations": 14406,
  "independent_nashpy_cases": 8,
  "cost_one_mixed_pairs": 121,
  "acceptance_cost_scenarios": 24,
  "settlements": [
    {
      "state": "accepted",
      "deposit": 3,
      "held": 0,
      "provider_payment": 3,
      "buyer_refund": 0,
      "provider_net": 2,
      "sponsor_check_and_appeal_cost": 0.25
    },
    {
      "state": "rejected-on-appeal",
      "deposit": 3,
      "held": 0,
      "provider_payment": 0,
      "buyer_refund": 3,
      "provider_net": -1,
      "sp